# ### Silver layer incremental 
Step 1 --Imports and setup 

In [0]:
from pyspark.sql import functions as F 
from pyspark.sql.window import Window 
from delta.tables import DeltaTable 
from datetime import   datetime 
import uuid 

In [0]:
spark.sql("  use catalog novacart_adb ")
spark.sql(" create schema if not exists silver_schema")
silver_run_id= str(uuid.uuid4())
print ("current silver  uuid ", silver_run_id)

### Step 2 --> Silver Control Table 
This table stores the latest silver processing state for each entity 


In [0]:
spark.sql("DROP TABLE IF EXISTS novacart_adb.silver_schema.processing_control")
print("dropped")


spark.sql("""
    CREATE TABLE IF NOT EXISTS novacart_adb.silver_schema.processing_control (
        layer                             STRING,
        entity_name                       STRING,
        last_processed_bronze_run_id      STRING,
        last_processed_bronze_ingested_at TIMESTAMP,
        rows_merged                       BIGINT,
        run_status                        STRING,
        silver_run_id                     STRING,
        updated_at                        TIMESTAMP
    )
    USING DELTA
""")
print("created")

### Step -3 Helper Functions 
usert_to_silver() merges cleaned/transformed  rows into Silver target table  
get_last_processed_bronze_ingested_at()  reads the silver watermark 
upsert_silver_control()  updates the  silver control table 
get_incremental_bronze() reads only new bronze rows  that silver hasnt processed  yet  

In [0]:
def upsert_to_silver(df_source,target_table, join_key):
    if spark.catalog.tableExists(target_table):
        dt= DeltaTable.forName(spark,target_table)
        dt.alias("target").merge(
            df_source.alias("source"),
            f"target.{join_key}= source.{join_key}"
        ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
    else:
        df_source.write.format("delta").saveAsTable(target_table)



In [0]:
def get_last_processed_bronze_ingested_at(entity_name: str):
    cntrl = (
        spark.table("novacart_adb.silver_schema.processing_control")
        .filter(
            (F.col("layer") == "silver") &
            (F.col("entity_name") == entity_name) &
            (F.col("run_status") == "Success")  
        )
        .orderBy(F.col("updated_at").desc())
        .limit(1)
    )
    rows = cntrl.collect()

    if not rows:
        return None

    return rows[0]["last_processed_bronze_ingested_at"]

In [0]:
import uuid
from delta.tables import DeltaTable
from datetime import datetime

silver_run_id = str(uuid.uuid4())
print("silver_run_id:", silver_run_id)

def upsert_silver_control(entity_name, last_processed_bronze_run_id,
                           last_processed_ingested_at, rows_merged):
    cntrl_df = spark.createDataFrame(
        [(
            "silver",
            entity_name,
            last_processed_bronze_run_id,
            last_processed_ingested_at,
            int(rows_merged),
            "Success",
            silver_run_id,
            datetime.utcnow(),
        )],
        schema="""
            layer                             STRING,
            entity_name                       STRING,
            last_processed_bronze_run_id      STRING,
            last_processed_bronze_ingested_at TIMESTAMP,
            rows_merged                       BIGINT,
            run_status                        STRING,
            silver_run_id                     STRING,
            updated_at                        TIMESTAMP
        """,
    )

    dt = DeltaTable.forName(spark, "novacart_adb.silver_schema.processing_control")

    (dt.alias("t")
     .merge(
         cntrl_df.alias("s"),
         "t.layer = s.layer AND t.entity_name = s.entity_name"
     )
     .whenMatchedUpdate(set={
         "last_processed_bronze_run_id":      "s.last_processed_bronze_run_id",
         "last_processed_bronze_ingested_at": "s.last_processed_bronze_ingested_at",
         "rows_merged":                       "s.rows_merged",
         "run_status":                        "s.run_status",
         "silver_run_id":                     "s.silver_run_id",
         "updated_at":                        "s.updated_at"
     })
     .whenNotMatchedInsertAll()
     .execute()
    )
    print(f"  Silver control updated for {entity_name}")

In [0]:
def get_incremental_bronze(bronze_table,entity_name):
    last_ingested_at=get_last_processed_bronze_ingested_at(entity_name)
    bronze_df=spark.read.table(bronze_table)
    if last_ingested_at is None:
        return bronze_df,last_ingested_at
    return bronze_df.filter(F.col("bronze_ingested_at")>F.lit(last_ingested_at)), last_ingested_at
    

# Orders Incremental processing 
--> reads only new bronze order rows
-->cleans values like order_status and order_amount 
--> keeps only the latest version per order_id
--> validate bussiness rule 
--> sends bad rows to quarantine 
--> merges good rows into orders_transformed

In [0]:
# ── STEP 1: Read incremental Bronze rows ──────────────────────
orders_inc, last_orders_ingested_at = get_incremental_bronze(
    "novacart_adb.bronze_schema.orders_raw",
    "orders"
)

orders_inc_count = orders_inc.count()
print(f"orders rows_to_process_in_silver = {orders_inc_count}")

if orders_inc_count > 0:

    # ── STEP 2: Deduplication window ──────────────────────────
    order_window = Window.partitionBy("order_id").orderBy(
        F.col("updated_at").cast("timestamp").desc(),
        F.col("bronze_ingested_at").desc()
    )

    # ── STEP 3: Clean the data ────────────────────────────────
    orders_cleaned = (
        orders_inc
        .withColumn("order_status",
            F.upper(F.trim(F.col("order_status"))))
        .withColumn("order_status",
            F.when(F.col("order_status") == "", F.lit(None))
            .otherwise(F.col("order_status")))
        .withColumn("order_amount",
            F.regexp_replace(F.col("order_amount"), r"[$, ]", ""))
        .withColumn("order_amount",
            F.when(
                F.trim(F.col("order_amount")).isin("N/A", "NULL", "??", ""),
                None
            ).otherwise(F.col("order_amount")))
        .withColumn("order_amount",
            F.col("order_amount").cast("double"))
        .withColumn("created_at", F.to_timestamp("created_at"))
        .withColumn("updated_at", F.to_timestamp("updated_at"))
        .withColumn("row_rank", F.row_number().over(order_window))
        .filter(F.col("row_rank") == 1)
        .drop("row_rank")
        .withColumn("silver_run_id", F.lit(silver_run_id))
    )

    # ── STEP 4: Validate the cleaned data ─────────────────────
    orders_validated = (
        orders_cleaned
        .withColumn(
            "to_be_verified_by_orders_team",
            F.when(F.col("customer_id").isNull(),
                "verify_customer_id")
            .when(F.col("product_id").isNull(),
                "verify_product_id")
            .when(
                F.col("order_status").isNull() |
                (F.trim(F.col("order_status")) == ""),
                "verify_order_status")
            .when(
                F.col("order_amount").isNull() |
                (F.col("order_amount") <= 0),
                "verify_order_amount")
            .otherwise("No Issues")
        )
        .withColumn(
            "check_order_amount",
            F.when(
                F.col("order_amount").isNull() |
                (F.col("order_amount") <= 0),
                F.lit(True)
            ).otherwise(F.lit(False))
        )
        .withColumn("order_date",  F.to_date("created_at"))
        .withColumn("order_year",  F.year("created_at"))
        .withColumn("order_month", F.month("created_at"))
        .withColumn("order_day",   F.dayofmonth("created_at"))
        .withColumn("order_dow",   F.date_format("created_at", "E"))
    )

    # ── STEP 5: Split good and bad rows ───────────────────────
    orders_good = orders_validated.filter(
        F.col("to_be_verified_by_orders_team") == "No Issues"
    )

    orders_bad = (
        orders_validated
        .filter(F.col("to_be_verified_by_orders_team") != "No Issues")
        .withColumn("quarantine_ts", F.current_timestamp())
    )

    print(f"orders_good count:  {orders_good.count()}")
    print(f"orders_bad count:   {orders_bad.count()}")

    # ── STEP 6: Write good rows to Silver via MERGE ───────────
    upsert_to_silver(
        orders_good,
        "novacart_adb.silver_schema.orders_transformed",
        "order_id"
    )

    # ── STEP 7: Write bad rows to quarantine ──────────────────
    orders_bad.write.format("delta").mode("append").saveAsTable(
        "novacart_adb.silver_schema.orders_quarantine"
    )

    # ── STEP 8: Calculate Silver watermark ────────────────────
    mx_ingested = (
        orders_inc
        .agg(F.max("bronze_ingested_at").alias("mx"))
        .collect()[0]["mx"]
    )

    mx_run = (
        orders_inc
        .filter(F.col("bronze_ingested_at") == F.lit(mx_ingested))
        .agg(F.max("bronze_run_id").alias("mx"))
        .collect()[0]["mx"]
    )

    # ── STEP 9: Update Silver control table ───────────────────
    upsert_silver_control(
        "orders",
        mx_run,
        mx_ingested,
        orders_good.count()
    )

else:
    print("No new orders Bronze rows for Silver.")
    upsert_silver_control(
        "orders",
        None,
        last_orders_ingested_at,
        orders_inc_count
    )

In [0]:
import inspect
print(inspect.getsource(upsert_silver_control))

In [0]:
%sql 
select * from novacart_adb.silver_schema.orders_quarantine;

In [0]:
display(orders_cleaned)

In [0]:
# ── STEP 1: Read incremental Bronze rows ──────────────────────
products_inc, last_products_ingested_at = get_incremental_bronze(
    "novacart_adb.bronze_schema.products_raw",
    "products"
)

products_inc_count = products_inc.count()
print(f"products rows_to_process_in_silver = {products_inc_count}")

if products_inc_count > 0:

    # ── STEP 2: Deduplication window ──────────────────────────
    product_window = Window.partitionBy("product_id").orderBy(
        F.col("updated_at").cast("timestamp").desc(),
        F.col("bronze_ingested_at").desc()
    )

    # ── STEP 3: Clean the data ────────────────────────────────
    products_cleaned = (
        products_inc

        # Standardize product_name — trim spaces and uppercase
        .withColumn("product_name",
            F.upper(F.trim(F.col("product_name"))))

        # Convert empty product_name to NULL
        .withColumn("product_name",
            F.when(F.col("product_name") == "", F.lit(None))
            .otherwise(F.col("product_name")))

        # Fix category — standardize casing and fix typos
        .withColumn("category",
            F.when(
                F.upper(F.trim(F.col("category"))).contains("ELECTRNICS"),
                "ELECTRONICS"                    # fix typo
            )
            .otherwise(F.upper(F.trim(F.col("category"))))
        )

        # Clean price — step by step
        .withColumn("price", F.trim(F.col("price")))
        .withColumn("price",
            F.regexp_replace(F.col("price"), r"\$", ""))  # remove $
        .withColumn("price",
            F.regexp_replace(F.col("price"), ",", "."))   # comma → dot
        .withColumn("price",
            F.regexp_replace(F.col("price"), r"\s+", "")) # remove spaces

        # Cast price to double using try_cast (returns NULL if fails)
        .withColumn("price",
            F.expr("try_cast(price as double)"))

        # Parse updated_at to timestamp
        .withColumn("updated_at", F.to_timestamp("updated_at"))

        # Deduplicate — keep latest record per product_id
        .withColumn("row_rank", F.row_number().over(product_window))
        .filter(F.col("row_rank") == 1)
        .drop("row_rank")

        # Tag with Silver run ID
        .withColumn("silver_run_id", F.lit(silver_run_id))
    )

    # ── STEP 4: Validate the cleaned data ─────────────────────
    products_validated = (
        products_cleaned

        # Flag reason for rejection
        .withColumn(
            "to_be_verified_by_products_team",
            F.when(F.col("product_name").isNull(),
                "verify_product_name")
            .when(F.col("category").isNull(),
                "verify_category")
            .when(
                F.col("price").isNull() | (F.col("price") <= 0),
                "verify_price")
            .otherwise("No Issues")
        )

        # Boolean flag for price issues
        .withColumn(
            "check_product_price",
            F.when(
                F.col("price").isNull() | (F.col("price") <= 0),
                "invalid_price"
            ).otherwise("valid_price")
        )
    )

    # ── STEP 5: Split good and bad rows ───────────────────────
    products_good = products_validated.filter(
        (F.col("to_be_verified_by_products_team") == "No Issues") &
        (F.col("check_product_price") == "valid_price")
    )

    # Drop price_raw if exists
    if "price_raw" in products_good.columns:
        products_good = products_good.drop("price_raw")

    products_bad = (
        products_validated
        .filter(
            (F.col("to_be_verified_by_products_team") != "No Issues") |
            (F.col("check_product_price") == "invalid_price")
        )
        .withColumn("quarantine_ts", F.current_timestamp())
    )

    print(f"products_good count: {products_good.count()}")
    print(f"products_bad count:  {products_bad.count()}")

    # ── STEP 6: Write good rows to Silver via MERGE ───────────
    upsert_to_silver(
        products_good,
        "novacart_adb.silver_schema.products_transformed",
        "product_id"
    )

    # ── STEP 7: Write bad rows to quarantine ──────────────────
    products_bad.write.format("delta").mode("append").saveAsTable(
        "novacart_adb.silver_schema.products_quarantine"
    )

    # ── STEP 8: Calculate Silver watermark ────────────────────
    mx_ingested = (
        products_inc
        .agg(F.max("bronze_ingested_at").alias("mx"))
        .collect()[0]["mx"]
    )

    mx_run = (
        products_inc
        .filter(F.col("bronze_ingested_at") == F.lit(mx_ingested))
        .agg(F.max("bronze_run_id").alias("mx"))
        .collect()[0]["mx"]
    )

    # ── STEP 9: Update Silver control table ───────────────────
    upsert_silver_control(
        "products",
        mx_run,
        mx_ingested,
        products_good.count()
    )

else:
    print("No new products Bronze rows for Silver.")
    upsert_silver_control(
        "products",
        None,
        last_products_ingested_at,
        products_inc_count
    )

In [0]:
%sql
select * from novacart_adb.silver_schema.products_transformed

In [0]:
# ── STEP 1: Read incremental Bronze rows ──────────────────────
payments_inc, last_payments_ingested_at = get_incremental_bronze(
    "novacart_adb.bronze_schema.payments_raw",
    "payments"
)
print("Payments last processed Bronze ingested_at =", last_payments_ingested_at)

payments_inc_count = payments_inc.count()
print(f"payments rows_to_process_in_silver = {payments_inc_count}")

if payments_inc_count > 0:

    # ── STEP 2: Deduplication window ──────────────────────────
    payment_window = Window.partitionBy("payment_id").orderBy(
        F.col("processed_at").cast("timestamp").desc(),
        F.col("bronze_ingested_at").desc()
    )

    # ── STEP 3: Clean the data ────────────────────────────────
    payments_cleaned = (
        payments_inc

        # Standardize payment_status to uppercase
        .withColumn("payment_status",
            F.upper(F.trim(F.col("payment_status"))))

        # Convert empty payment_status to NULL
        .withColumn("payment_status",
            F.when(F.col("payment_status") == "", F.lit(None))
            .otherwise(F.col("payment_status")))

        # Clean paid_amount step by step
        .withColumn("paid_amount", F.trim(F.col("paid_amount")))
        .withColumn("paid_amount",
            F.regexp_replace(F.col("paid_amount"), r"\$", ""))
        .withColumn("paid_amount",
            F.regexp_replace(F.col("paid_amount"), ",", "."))
        .withColumn("paid_amount",
            F.regexp_replace(F.col("paid_amount"), r"\s+", ""))

        # Cast to double safely
        .withColumn("paid_amount",
            F.expr("try_cast(paid_amount as double)"))

        # Parse processed_at to timestamp
        .withColumn("processed_at",
            F.to_timestamp("processed_at"))

        # Deduplicate — keep latest per payment_id
        .withColumn("row_rank", F.row_number().over(payment_window))
        .filter(F.col("row_rank") == 1)
        .drop("row_rank")

        # Tag with Silver run ID
        .withColumn("silver_run_id", F.lit(silver_run_id))
    )

    # ── STEP 4: Validate ──────────────────────────────────────
    payments_validated = (
        payments_cleaned

        # Flag reason for rejection
        .withColumn(
            "to_be_verified_by_payments_team",
            F.when(F.col("order_id").isNull(),
                "verify_order_id")
            .when(F.col("payment_status").isNull(),
                "verify_payment_status")
            .when(
                F.col("paid_amount").isNull() |
                (F.col("paid_amount") <= 0),
                "verify_paid_amount")
            .otherwise("No Issues")
        )

        # Boolean flag for amount issues
        .withColumn(
            "check_paid_amount",
            F.when(
                F.col("paid_amount").isNull() |
                (F.col("paid_amount") <= 0),
                F.lit(True)
            ).otherwise(F.lit(False))
        )
    )

    # ── STEP 5: Split good and bad rows ───────────────────────
    payments_good = payments_validated.filter(
        F.col("to_be_verified_by_payments_team") == "No Issues"
    )

    payments_bad = (
        payments_validated
        .filter(
            F.col("to_be_verified_by_payments_team") != "No Issues"
        )
        .withColumn("quarantine_ts", F.current_timestamp())
    )

    print(f"payments_good count: {payments_good.count()}")
    print(f"payments_bad count:  {payments_bad.count()}")

    # ── STEP 6: Write good rows to Silver via MERGE ───────────
    upsert_to_silver(
        payments_good,
        "novacart_adb.silver_schema.payments_transformed",
        "payment_id"
    )

    # ── STEP 7: Write bad rows to quarantine ──────────────────
    payments_bad.write.format("delta").mode("append").saveAsTable(
        "novacart_adb.silver_schema.payments_quarantine"
    )

    # ── STEP 8: Calculate Silver watermark ────────────────────
    mx_ingested = (
        payments_inc
        .agg(F.max("bronze_ingested_at").alias("mx"))
        .collect()[0]["mx"]
    )

    mx_run = (
        payments_inc
        .filter(F.col("bronze_ingested_at") == F.lit(mx_ingested))
        .agg(F.max("bronze_run_id").alias("mx"))
        .collect()[0]["mx"]
    )

    # ── STEP 9: Update Silver control table ───────────────────
    upsert_silver_control(
        "payments",
        mx_run,
        mx_ingested,
        payments_good.count()
    )

else:
    print("No new payments Bronze rows for Silver.")
    upsert_silver_control(
        "payments",
        None,
        last_payments_ingested_at,
        payments_inc_count
    )

In [0]:
print("Products transformed count:", 
    spark.sql("SELECT COUNT(*) FROM novacart_adb.silver_schema.products_transformed")
    .collect()[0][0])

print("Orders transformed count:", 
    spark.sql("SELECT COUNT(*) FROM novacart_adb.silver_schema.orders_transformed")
    .collect()[0][0])

print("Payments transformed count:", 
    spark.sql("SELECT COUNT(*) FROM novacart_adb.silver_schema.payments_transformed")
    .collect()[0][0])

display(
    spark.table("novacart_adb.silver_schema.processing_control")
    .orderBy("entity_name")
)